# Урок 2.2. Векторные базы данных: зачем они и как работают

**Модуль 2 «Embeddings и векторные базы данных» · Курс «LLM для Python-разработчиков»**

В прошлом уроке наш семантический поиск сравнивал запрос с *каждым* документом. На 14 документах «Векторики» это незаметно. Сегодня мы честно упрёмся в масштаб — сгенерируем сто тысяч векторов, замерим полный перебор, а потом **построим два ANN-индекса своими руками** и увидим главный компромисс индустрии: скорость в обмен на полноту. И только поняв механику — выберем себе базу данных.

## 🎯 Цели урока

После этого урока вы сможете:

- оценивать, **когда полный перебор перестаёт работать** (и когда его, наоборот, достаточно);
- объяснить идею **ANN** (approximate nearest neighbors) и метрику **recall@k**;
- рассказать «на пальцах» — и показать кодом — как работают два главных семейства индексов: **разбиение пространства (IVF)** и **графы малого мира (HNSW)**;
- крутить ручки **nprobe / ef** осознанно: что растёт, что падает;
- выбирать между **FAISS, Chroma, Qdrant и pgvector** под конкретный проект;
- знать ловушку **фильтров по метаданным** в векторном поиске.

## План

1. Проблема: O(N) на каждый запрос
2. ANN и recall@k: жертвуем полнотой ради скорости
3. Идея №1 — разбить пространство (мини-IVF своими руками)
4. Идея №2 — граф малого мира (мини-HNSW своими руками)
5. Что такое векторная БД (и чем она больше, чем индекс)
6. FAISS vs Chroma vs Qdrant vs pgvector

⏱ **Время:** ~75–90 минут.

> 💡 Весь урок работает сразу: только numpy, никаких моделей и скачиваний. Числа замеров на вашей машине будут своими — в этом и смысл.

## 1. Проблема: O(N) на каждый запрос

Вспомните устройство поиска из урока 2.1: косинус запроса с каждым вектором, сортировка, топ-k. Это **линейный скан** — как `SELECT ... WHERE` по таблице без индекса. Каждый запрос пользователя стоит O(N × dim) умножений.

Прочувствуем на числах. Сгенерируем синтетический «датасет эмбеддингов»: 100 000 векторов по 384 измерения — как если бы мы проиндексировали стотысячную базу знаний моделью из урока 2.1. Важная деталь: настоящие эмбеддинги не рассыпаны равномерно — тексты группируются по темам. Сымитируем это: 200 «тем», вокруг каждой — облако точек:

In [ ]:
import time

import numpy as np

rng = np.random.default_rng(42)

N, DIM, N_TOPICS = 100_000, 384, 200

# Разброс точек внутри темы. Калибровка нетривиальна: в 384 измерениях норма
# гауссовского шума ~ SIGMA * sqrt(384). При SIGMA=0.05 шум сравним с длиной
# центра темы — темы получаются плотными облаками (косинус внутри темы ~0.5),
# но не точками. Поставьте 0.35 — и шум задавит сигнал в 7 раз: "темы" исчезнут,
# а с ними и вся польза индексов. Проверьте после урока!
SIGMA = 0.05

# центры "тем" на единичной сфере
topic_centers = rng.standard_normal((N_TOPICS, DIM), dtype=np.float32)
topic_centers /= np.linalg.norm(topic_centers, axis=1, keepdims=True)

# каждый вектор = центр своей темы + шум, затем нормировка (как в уроке 2.1)
topic_of = rng.integers(0, N_TOPICS, N)
vectors = topic_centers[topic_of] + SIGMA * rng.standard_normal((N, DIM), dtype=np.float32)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

print(f"База: {vectors.shape}, память: {vectors.nbytes / 2**20:.0f} МБ")

# 100 поисковых запросов из тех же тем (но это НОВЫЕ точки, не из базы)
N_QUERIES = 100
query_topics = rng.integers(0, N_TOPICS, N_QUERIES)
queries = topic_centers[query_topics] + SIGMA * rng.standard_normal((N_QUERIES, DIM), dtype=np.float32)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

# --- полный перебор: по одному запросу, как они приходят в проде ---
start = time.perf_counter()
for query in queries:
    sims = vectors @ query                       # косинусы со ВСЕЙ базой
    top10 = np.argpartition(sims, -10)[-10:]     # топ-10 без полной сортировки
brute_ms = (time.perf_counter() - start) / N_QUERIES * 1000
print(f"Полный перебор: {brute_ms:.2f} мс на запрос при N={N:,}")

print(f"\n{'N векторов':>12} {'мс/запрос':>12} {'память':>10}")
for scale in (10**5, 10**6, 10**7, 10**8):
    est_ms = brute_ms * scale / N
    mem_gb = scale * DIM * 4 / 2**30
    print(f"{scale:>12,} {est_ms:>12,.0f} {mem_gb:>9.1f}Г")

Полный перебор растёт **линейно**: и время, и память. На ста тысячах — миллисекунды, всё отлично. На десяти миллионах — уже сотни миллисекунд *на один запрос*: при 50 RPS вам понадобятся десятки ядер только на поиск, а векторы перестанут влезать в память одной машины.

Честная оговорка, которую пропускают продавцы векторных БД: **до ~100 тысяч векторов перебор — прекрасное решение.** Recall = 100%, кода — три строки numpy, инфраструктуры — ноль. Не тащите Qdrant в прототип на пять тысяч документов.

Но дальше нужен индекс. И тут сюрприз для бэкендера: **точного** индекса для «ближайших соседей» в сотнях измерений не существует — проклятие размерности съедает деревья и хеши, которые прекрасно работают в 2–3 измерениях. Индустрия сдалась и сменила вопрос:

## 2. ANN: жертвуем полнотой ради скорости

**ANN** (approximate nearest neighbors) — «найди *почти* наверняка *почти* всех ближайших соседей, но за миллисекунды». Насколько «почти» — измеримо, и метрика вам уже знакома по духу из урока 1.7:

**recall@k** = какая доля из k истинных ближайших соседей (по полному перебору) попала в выдачу индекса. Recall 0.95@10 — «в среднем 9.5 из настоящей десятки на месте».

Посчитаем эталон (ground truth) один раз — с ним будем сравнивать все наши индексы:

In [ ]:
# эталон: истинный топ-10 для каждого запроса (полный перебор, офлайн можно батчем)
all_sims = queries @ vectors.T                        # (100, 100000)
true_top10 = np.argpartition(all_sims, -10, axis=1)[:, -10:]

def recall_at_10(found_ids, query_idx: int) -> float:
    """Доля истинного топ-10, попавшая в выдачу."""
    return len(set(map(int, found_ids)) & set(map(int, true_top10[query_idx]))) / 10

print("Эталон готов: истинный топ-10 для каждого из 100 запросов.")
print("Проверка на самом переборе:",
      recall_at_10(np.argpartition(vectors @ queries[0], -10)[-10:], 0))

## 3. Идея №1: разбить пространство (IVF)

Как вы ищете кафе? Не перебираете все заведения города — вы смотрите **в своём районе** и, может, в паре соседних. Первая великая идея ANN ровно такая:

```text
1. Офлайн: разбить все векторы на "районы" (кластеры) — например, k-means.
   У каждого района есть центр (centroid).
2. Запрос: найти nprobe ближайших ЦЕНТРОВ (их всего сотни — быстро),
   и перебрать только векторы этих районов.
```

Это семейство **IVF** (inverted file index — «инвертированный список»: район → его векторы). Ручка компромисса — **nprobe**: сколько районов проверять. Больше — точнее и медленнее.

Строим руками. Мини-k-means в десять строк (профессиональный был бы хитрее, но механика та же):

In [ ]:
K_CLUSTERS = 400   # "районов" больше, чем тем: у кривой nprobe будет наклон

def mini_kmeans(data: np.ndarray, k: int, iters: int = 5, seed: int = 0):
    """Простейший k-means на нормированных векторах (близость = косинус)."""
    km_rng = np.random.default_rng(seed)
    centroids = data[km_rng.choice(len(data), k, replace=False)].copy()
    for _ in range(iters):
        assign = np.argmax(data @ centroids.T, axis=1)   # ближайший центр
        for c in range(k):
            members = data[assign == c]
            if len(members):
                center = members.mean(axis=0)
                centroids[c] = center / np.linalg.norm(center)
    return centroids, assign

start = time.perf_counter()
centroids, assignments = mini_kmeans(vectors, K_CLUSTERS)
build_s = time.perf_counter() - start

# инвертированный список: номер района -> индексы его векторов
cluster_members = [np.where(assignments == c)[0] for c in range(K_CLUSTERS)]
sizes = [len(m) for m in cluster_members]
print(f"Индекс построен за {build_s:.1f} с; районов: {K_CLUSTERS}, "
      f"размер района: от {min(sizes)} до {max(sizes)} векторов")

def ivf_search(query: np.ndarray, nprobe: int = 5, k: int = 10) -> np.ndarray:
    """Поиск: nprobe ближайших районов -> перебор только их жителей."""
    top_cells = np.argpartition(query @ centroids.T, -nprobe)[-nprobe:]
    candidates = np.concatenate([cluster_members[c] for c in top_cells])
    if len(candidates) <= k:      # районы бывают и пустыми - k-means не идеален
        return candidates
    sims = vectors[candidates] @ query
    best_local = np.argpartition(sims, -k)[-k:]
    return candidates[best_local]

# замер: скорость и recall при nprobe=2
start = time.perf_counter()
recalls = [recall_at_10(ivf_search(q, nprobe=2), i) for i, q in enumerate(queries)]
ivf_ms = (time.perf_counter() - start) / N_QUERIES * 1000

print(f"\nIVF (nprobe=2):  {ivf_ms:.2f} мс/запрос, recall@10 = {np.mean(recalls):.2f}")
print(f"Полный перебор:  {brute_ms:.2f} мс/запрос, recall@10 = 1.00")
print(f"Ускорение: ~{brute_ms / ivf_ms:.0f}x — за счёт просмотра ~{2 / K_CLUSTERS:.1%} базы")

Вот он, компромисс ANN в одной строке вывода: **на порядок быстрее при recall под единицу** — мы просмотрели полпроцента базы, а нашли почти всех истинных соседей, потому что похожие векторы лежат в одних районах. Ручку `nprobe` вы покрутите в упражнении 1 и увидите всю кривую «скорость ↔ полнота».

У IVF есть слабое место: если истинный сосед оказался *сразу за границей* просмотренного района — мы его не увидим. Вторая великая идея решает это иначе.

## 4. Идея №2: граф малого мира (HNSW)

Знакомьтесь с индексом, который стоит почти во всех современных векторных БД: **HNSW** (Hierarchical Navigable Small World). Идея в двух шагах.

**Шаг 1 — навигация по графу знакомств.** Соедините каждый вектор рёбрами с M ближайшими соседями. Поиск — «испорченный телефон»: встаём в произвольную вершину и жадно шагаем к тому соседу, который ближе к запросу. Как передать письмо незнакомцу через цепочку знакомых — эффект «шести рукопожатий» (small world).

**Шаг 2 — иерархия, как в skip-list.** Жадному поиску с случайного старта долго идти через весь граф мелкими шагами. Поэтому над основным графом строят «этажи» из случайного подмножества точек с длинными связями — редеющие кверху. Если вы работали с Redis sorted sets — это буквально та же идея, что skip-list, только «дальше/ближе» вместо «больше/меньше»:

```text
слой 2:   A ──────────────── K            мало точек, длинные прыжки
          │                  │
слой 1:   A ────── F ─────── K ──── P     точек больше, прыжки короче
          │        │         │      │
слой 0:   A─B─C─D─E─F─G─H─I─J─K─...─P     все точки, локальные связи

Поиск: входим на верхнем слое, жадно доходим до ближайшей к запросу
вершины, спускаемся на слой ниже, уточняем. O(log N) шагов.
```

Построим такой мир руками на базе из 5 000 векторов. Первая попытка — самая интуитивная: соединим каждую вершину с её M ближайшими соседями, и всё:

In [ ]:
import heapq

N_SMALL = 5_000
vectors_small = vectors[:N_SMALL]
true_small = np.argpartition(queries @ vectors_small.T, -10, axis=1)[:, -10:]

# граф: каждую вершину соединяем с M БЛИЖАЙШИМИ (офлайн, честный перебор)
M = 16
start = time.perf_counter()
sims_all = vectors_small @ vectors_small.T
np.fill_diagonal(sims_all, -np.inf)
nearest_graph = np.argpartition(sims_all, -M, axis=1)[:, -M:]
del sims_all
print(f"Граф построен за {time.perf_counter() - start:.1f} с: "
      f"{N_SMALL} вершин, по {M} рёбер к ближайшим")

def nsw_search(graph, query: np.ndarray, k: int = 10, ef: int = 16, entry: int = 0):
    """Жадный поиск по графу с "лучом" из ef кандидатов (как слой HNSW).

    ef — ширина луча: сколько лучших кандидатов удерживаем и развиваем.
    Возвращает (топ-k, сколько вершин посмотрели).
    """
    start_sim = float(vectors_small[entry] @ query)
    visited = {entry}
    frontier = [(-start_sim, entry)]      # куча "кого развивать" (макс по sim)
    best = [(start_sim, entry)]           # куча "лучшие найденные" (мин наверху)
    while frontier:
        neg_sim, node = heapq.heappop(frontier)
        if len(best) >= ef and -neg_sim < best[0][0]:
            break                          # кандидаты стали хуже худшего из best
        for neighbor in graph[node]:
            neighbor = int(neighbor)
            if neighbor in visited:
                continue
            visited.add(neighbor)
            sim = float(vectors_small[neighbor] @ query)
            if len(best) < ef or sim > best[0][0]:
                heapq.heappush(best, (sim, neighbor))
                if len(best) > ef:
                    heapq.heappop(best)
                heapq.heappush(frontier, (-sim, neighbor))
    top = [node for _, node in sorted(best, reverse=True)[:k]]
    return top, len(visited)

def eval_graph(graph, ef: int) -> tuple[float, float]:
    recalls, looked = [], []
    for i, query in enumerate(queries):
        found, n_visited = nsw_search(graph, query, ef=ef)
        recalls.append(len(set(found) & set(map(int, true_small[i]))) / 10)
        looked.append(n_visited)
    return float(np.mean(recalls )), float(np.mean(looked))

for ef in (16, 64):
    recall, looked = eval_graph(nearest_graph, ef)
    print(f"ef={ef:<3} recall@10 = {recall:.2f}, просмотрено вершин: {looked:.0f}")

Катастрофа: recall около нуля — и посмотрите на число просмотренных вершин: **ровно ~25, сколько ни поднимай ef**. Это не совпадение: 5 000 векторов на 200 тем — по 25 точек в теме. Граф «только ближайшие» развалился на **200 изолированных островов**: все 16 рёбер каждой вершины ведут к соседям по теме, и жадный поиск, стартовав на чужом острове, обошёл его целиком и остановился — мостов на другие острова нет.

Это фундаментальный факт про навигируемые графы, известный как модель «малого мира» Уоттса—Строгаца: **для навигации нужны и короткие связи, и редкие длинные «мосты»**. Как в жизни: чтобы передать письмо на другой континент, нужны не только соседи по подъезду, но и пара дальних знакомых. Добавим мосты — заменим часть ближних рёбер на случайные дальние:

In [ ]:
N_BRIDGES = 6
bridge_rng = np.random.default_rng(1)

# 10 рёбер к ближайшим + 6 случайных "мостов" из каждой вершины
small_world_graph = np.concatenate(
    [nearest_graph[:, : M - N_BRIDGES],
     bridge_rng.integers(0, N_SMALL, (N_SMALL, N_BRIDGES))],
    axis=1,
)

print(f"{'ef':>4} {'recall@10':>10} {'просмотрено':>12}")
for ef in (16, 64, 128):
    recall, looked = eval_graph(small_world_graph, ef)
    print(f"{ef:>4} {recall:>10.2f} {looked:>8.0f} из {N_SMALL}")

Те же вершины, тот же жадный поиск — но шесть случайных мостов превратили архипелаг в **малый мир**: recall взлетел с 0.02 к единице, при просмотре ~10–20% вершин. И вторая ручка компромисса перед вами — **ef** (ширина луча): больше кандидатов → выше recall → больше просмотров.

Настоящий HNSW делает мосты не случайными, а **структурными** — это и есть его этажи: верхние слои с длинными связями (магистрали между «континентами»), нижние — с короткими (улицы). Плюс эвристики отбора рёбер при построении. Но физику вы только что увидели своими глазами: *локальные связи находят точное место, длинные — дают до него добраться*. Обе ручки теперь знакомы:

| параметр | на что влияет | когда крутить |
|---|---|---|
| `M` — рёбер на вершину | связность графа, память, качество | обычно 16–48, задаётся при создании |
| `ef_construction` | тщательность построения (офлайн) | 100–400; больше = дольше строим, лучше граф |
| `ef_search` | ширина луча при поиске | главная ручка «recall ↔ latency» в рантайме |

## 5. Векторная БД — это индекс плюс всё остальное

Индекс мы уже умеем. Почему бы просто не взять библиотеку индексов (например, **FAISS** от Meta) и не жить счастливо? Можно! Но вспомните, чем СУБД отличается от B-дерева в памяти. Векторная БД поверх индекса даёт:

- **CRUD**: вставка, обновление и удаление векторов без пересборки индекса (в HNSW удаление — нетривиальный трюк с тумбстоунами, хорошо, что не вам его писать);
- **метаданные и фильтры**: «найди похожее, но только среди документов отдела HR не старше года»;
- **персистентность**: векторы и граф переживают рестарт;
- **консистентность и API**: клиенты, аутентификация, коллекции;
- **масштабирование**: шардирование, репликация, снапшоты;
- **сжатие** (quantization — знакомое слово из урока 1.3? та же идея: float32 → int8, память минус 75%).

Про фильтры — отдельная ловушка, о которой стоит знать до выбора БД. Фильтровать можно **до** поиска (pre-filter: сначала отобрать подходящие по метаданным, потом искать среди них) или **после** (post-filter: найти топ-k, выкинуть неподходящие). Post-filter коварен: если условию удовлетворяет 5% базы, в топ-10 может не выжить *ни одного* результата. Прочувствуете в упражнении 3. Зрелые БД (Qdrant) умеют фильтровать *прямо во время* обхода графа — это дорогого стоит.

## 6. Выбираем: FAISS vs Chroma vs Qdrant vs pgvector

| | **FAISS** | **Chroma** | **Qdrant** | **pgvector** |
|---|---|---|---|---|
| что это | библиотека индексов | embedded-БД | БД-сервер | расширение Postgres |
| аналогия | «B-дерево в коробке» | SQLite | PostgreSQL | индекс в вашем PG |
| запуск | `import faiss` | `pip install chromadb` | Docker / сервер | `CREATE EXTENSION` |
| фильтры | пишете сами | есть, простые | мощные, внутри обхода графа | весь SQL `WHERE` |
| комфортный масштаб | любой (но всё сами) | до ~1 млн | миллионы—миллиарды | до ~1 млн |
| ops-нагрузка | вы и есть ops | нулевая | обычная серверная | у вас уже есть DBA |
| когда брать | своя ML-платформа | прототипы, ноутбуки, мелкий прод | продакшен-поиск как сервис | данные уже в PG, не хотите новый сервис |

Решается всё, как обычно, не модой, а обстоятельствами:

- **прототип, ноутбук, до сотен тысяч векторов** → Chroma: ноль инфраструктуры, следующий урок — про неё;
- **отдельный поисковый сервис в проде, фильтры, рост** → Qdrant: урок 2.4, докер-компоуз уже лежит в папке модуля;
- **у вас Postgres и миллион векторов** → pgvector: не плодите сервисы без нужды;
- **вы строите свою ML-инфраструктуру** → FAISS и команда, которая его приготовит.

Есть и managed-облака (Pinecone, Qdrant Cloud, Weaviate Cloud) — тот же выбор «облако vs своё», что мы считали в уроке 1.3, только для поиска.

## ⚠️ Частые ошибки

1. **Тащить векторную БД в прототип на 3 000 векторов.** Перебор numpy быстрее сетевого вызова к Qdrant. Индекс — решение проблемы масштаба; сначала убедитесь, что она у вас есть.
2. **Не мерить recall после включения ANN.** «Поиск стал каким-то странным» месяцами живёт в проде именно так. Эталон на сотне запросов делается за минуту — вы сегодня сделали.
3. **Дефолтные параметры навсегда.** Жалобы на качество → поднять `ef_search`/`nprobe` (за latency); жалобы на скорость → наоборот. Это ручка, а не константа.
4. **Post-filter без запаса.** Отфильтровали топ-10 по редкому условию — получили пустую выдачу. Либо запрашивайте k с запасом, либо берите БД с фильтрацией внутри поиска.
5. **Забыть про память.** Миллион векторов по 1024 измерения — это 4 ГБ float32 плюс граф сверху. Размерность модели эмбеддингов — это ещё и счёт за RAM (упражнение 2).
6. **Пересоздавать индекс на каждую вставку.** HNSW инкрементален — точки добавляются на лету. Пересборка нужна редко (смена модели — урок 2.1, помните?).

## 🏋️ Упражнения

### Упражнение 1 — кривая «скорость ↔ полнота» 📈

Прогоните `ivf_search` для `nprobe` из {1, 2, 5, 10, 20, 50}: таблица recall@10, мс/запрос, ускорение против перебора. Найдите «sweet spot»: минимальный nprobe с recall ≥ 0.95. Столько стоит несжимаемая честность выдачи.

### Упражнение 2 — цена размерности 💾

Модели эмбеддингов бывают на 128, 384 (наша) и 1024+ измерений. Для фиксированного N замерьте полный перебор на этих размерностях и посчитайте память базы (N × dim × 4 байта). Вывод сформулируйте как правило для выбора модели эмбеддингов.

### Упражнение 3 — проклятие post-фильтра 🕳

Раздайте всем 100 000 векторов случайную категорию из 20 (по ~5% на каждую). Сценарий: пользователь ищет топ-10 «только в своей категории». Посчитайте: сколько результатов в среднем выживает, если фильтровать *после* поиска топ-10? А сколько кандидатов надо запрашивать, чтобы стабильно набирать десятку? Сравните с pre-filter (поиск только по векторам категории).

## 📝 Мини-квиз

**1. У вас 40 000 документов и 2 запроса в секунду. Нужна ли векторная БД?**

<details><summary>Ответ</summary>

Скорее всего нет: полный перебор на таком объёме — единицы миллисекунд, recall 100%, инфраструктуры ноль. Векторная БД решает проблемы масштаба (миллионы векторов, высокий RPS, фильтры, персистентность) — если их нет, это преждевременная оптимизация.

</details>

**2. Что именно приносится в жертву при переходе на ANN-индекс и как за этим следить?**

<details><summary>Ответ</summary>

Полнота выдачи: индекс может пропустить часть истинных ближайших соседей. Метрика — recall@k против эталона полного перебора на выборке запросов. Замерили до включения, следите после изменения параметров.

</details>

**3. Объясните HNSW через skip-list.**

<details><summary>Ответ</summary>

Оба строят иерархию «редеющих кверху» уровней для быстрой навигации: skip-list — по порядку значений, HNSW — по близости векторов. Поиск входит на верхнем разреженном слое, большими прыжками доходит до окрестности цели, спускается и уточняет. Итог — O(log N) шагов вместо O(N).

</details>

**4. Пользователи жалуются на качество поиска. Какую ручку крутить у HNSW и чем заплатите?**

<details><summary>Ответ</summary>

`ef_search` вверх: шире луч кандидатов → выше recall. Платите латентностью (больше вершин просматривается на запрос). Аналог в IVF — `nprobe`.

</details>

**5. Команда уже держит PostgreSQL, векторов будет ~200 тысяч. Qdrant или pgvector?**

<details><summary>Ответ</summary>

Начать с pgvector: масштаб комфортный, фильтры — обычный SQL, новый сервис не нужен, бэкапы и мониторинг уже есть. Qdrant станет оправдан при миллионах векторов, тяжёлых фильтрованных запросах или выделении поиска в отдельный сервис.

</details>

## ✅ Решения упражнений

### Решение 1 — кривая «скорость ↔ полнота»

In [ ]:
print(f"{'nprobe':>7} {'recall@10':>10} {'мс/запрос':>11} {'ускорение':>10}")
sweet_spot = None
for nprobe in (1, 2, 5, 10, 20, 50):
    start = time.perf_counter()
    recalls = [recall_at_10(ivf_search(q, nprobe=nprobe), i)
               for i, q in enumerate(queries)]
    ms = (time.perf_counter() - start) / N_QUERIES * 1000
    recall = float(np.mean(recalls))
    print(f"{nprobe:>7} {recall:>10.2f} {ms:>11.2f} {brute_ms / ms:>9.1f}x")
    if sweet_spot is None and recall >= 0.95:
        sweet_spot = nprobe

if sweet_spot is not None:
    print(f"\nSweet spot: nprobe={sweet_spot} — минимальная цена за recall >= 0.95")
else:
    print("\nRecall 0.95 не достигнут — расширяйте поиск дальше")

Классическая кривая с насыщением: первые районы дают основной recall, дальше каждый процент полноты дорожает. Продакшен-выбор — точка на этой кривой, а не «дефолт из туториала». Обратите внимание: даже nprobe=50 (полбазы!) быстрее перебора — кандидатов меньше и куски памяти удобнее.

### Решение 2 — цена размерности

In [ ]:
N_TEST, N_Q = 50_000, 20

print(f"{'dim':>6} {'мс/запрос':>11} {'память базы':>12} {'на 10 млн':>10}")
for dim in (128, 384, 1024):
    data = rng.standard_normal((N_TEST, dim), dtype=np.float32)
    data /= np.linalg.norm(data, axis=1, keepdims=True)
    test_queries = rng.standard_normal((N_Q, dim), dtype=np.float32)
    test_queries /= np.linalg.norm(test_queries, axis=1, keepdims=True)

    start = time.perf_counter()
    for query in test_queries:
        top = np.argpartition(data @ query, -10)[-10:]
    ms = (time.perf_counter() - start) / N_Q * 1000

    mem_mb = N_TEST * dim * 4 / 2**20
    mem_10m_gb = 10**7 * dim * 4 / 2**30
    print(f"{dim:>6} {ms:>11.2f} {mem_mb:>10.0f}МБ {mem_10m_gb:>9.1f}Г")

del data

И время, и память растут линейно с размерностью. Правило для выбора модели эмбеддингов: **размерность — это не «чем больше, тем умнее», а тариф на каждый вектор навсегда**. Прирост качества между 384 и 1024 измерениями стоит утроения RAM и латентности — сверяйтесь с задачей (и с MTEB-рейтингом из урока 2.1), а не с максимализмом.

### Решение 3 — проклятие post-фильтра

In [ ]:
N_CATEGORIES = 20
categories = rng.integers(0, N_CATEGORIES, N)   # ~5% базы на категорию

# --- post-filter: ищем топ-10 по всей базе, потом выкидываем чужие категории
survivors = []
for i, query in enumerate(queries):
    target_cat = int(query_topics[i]) % N_CATEGORIES     # "категория пользователя"
    top10 = np.argpartition(vectors @ query, -10)[-10:]
    survivors.append(int(np.sum(categories[top10] == target_cat)))
print(f"Post-filter топ-10: выживает в среднем {np.mean(survivors):.1f} из 10 результатов")
print(f"Пустая выдача: в {np.mean(np.array(survivors) == 0):.0%} запросов")

# --- сколько кандидатов нужно, чтобы стабильно набрать десятку своих?
for fetch_k in (10, 50, 200, 400):
    got = []
    for i, query in enumerate(queries):
        target_cat = int(query_topics[i]) % N_CATEGORIES
        top = np.argpartition(vectors @ query, -fetch_k)[-fetch_k:]
        got.append(min(int(np.sum(categories[top] == target_cat)), 10))
    print(f"  запрашиваем {fetch_k:>3} кандидатов -> своих в среднем {np.mean(got):.1f} из 10")

# --- pre-filter: ищем сразу только среди своей категории
start = time.perf_counter()
for i, query in enumerate(queries):
    target_cat = int(query_topics[i]) % N_CATEGORIES
    subset = np.where(categories == target_cat)[0]
    sims = vectors[subset] @ query
    top10 = subset[np.argpartition(sims, -10)[-10:]]
pre_ms = (time.perf_counter() - start) / N_QUERIES * 1000
print(f"\nPre-filter: всегда ровно 10 своих, {pre_ms:.2f} мс/запрос "
      f"(ищем в 5% базы)")

Числа говорят сами: при категории в 5% базы post-filter топ-10 приносит **в среднем полрезультата**, а чтобы стабильно набрать десятку, кандидатов надо запрашивать в десятки раз больше (oversampling) — и это ломает всю экономию индекса. Pre-filter честнее, но требует, чтобы БД умела быстро находить подмножество. Лучшие движки (Qdrant) фильтруют прямо во время обхода HNSW-графа — теперь вы знаете, почему эта строчка в таблице сравнения стоила отдельного упоминания.

## 🎓 Итоги

- Полный перебор линеен: прекрасен до ~100k векторов, безнадёжен на миллионах. Точных индексов для сотен измерений нет — есть **ANN**: скорость в обмен на **recall@k**, который обязателен к измерению.
- **IVF** = разбить пространство на районы и смотреть ближайшие (`nprobe`). **HNSW** = граф малого мира со skip-list-иерархией и жадным поиском (`ef_search`). Обе механики вы сегодня написали руками и замерили.
- Главный инсайт дня: граф из одних ближайших соседей — **архипелаг островов**, жадный поиск в нём тонет. Навигируемость дают **длинные мосты** (у HNSW — верхние слои иерархии). Шесть случайных мостов подняли recall с 0.02 до ~1.0.
- Векторная БД = индекс + CRUD, фильтры, персистентность, репликация, сжатие. Фильтры — с ловушкой: post-filter опустошает выдачу, спасают oversampling, pre-filter или фильтрация внутри обхода.
- Выбор: **Chroma** — SQLite мира векторов (прототипы), **Qdrant** — PostgreSQL (прод), **pgvector** — «у нас уже есть Postgres», **FAISS** — библиотека для тех, кто строит своё.

**Дальше — урок 2.3 «Создание и поиск по эмбеддингам с Chroma»:** первая настоящая векторная БД в нашем стеке — заведём коллекции, метаданные и фильтры для базы знаний «Векторики».

## 📚 Что почитать

- [Оригинальная статья HNSW (Malkov, Yashunin)](https://arxiv.org/abs/1603.09320) — на удивление читаемая
- [Серия Pinecone про ANN-индексы](https://www.pinecone.io/learn/series/faiss/) — лучшие иллюстрации IVF/HNSW/PQ
- [Документация FAISS](https://github.com/facebookresearch/faiss/wiki) — энциклопедия индексов
- [Qdrant: Filtrable HNSW](https://qdrant.tech/articles/filtrable-hnsw/) — как устроена фильтрация внутри графа
- [ann-benchmarks.com](https://ann-benchmarks.com/) — независимые бенчмарки ANN-алгоритмов: те самые кривые recall↔QPS